In [1]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [2]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Per-Dataset Metadata

In [3]:
in_staging = True
profile = 'CPGnew'

In [4]:
project_name = "cpg0046-microrna"
source_list = ['edinburgh_igc']
output_parent_directory = "/Users/eweisbar/Desktop/cpg0046"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
# if unknown, comment out
per_dataset_manual = {
    'Plate_Size':384,
    'CP_Version':'v2',
    'DOI_to_Cite':'10.1101/2025.11.14.687149',
    'Year_Imaged': 2019,
    #'Cell_Line_Name':'',
    #'Cell_Line_Type':'',
    #'Cell_Line_Modification':'',
    'Microscope_Name':'Molecular Devices ImageXpress Micro XL',
    'Microscope_Binning': 1,
    'Microscope_Modality':'Widefield',
    'Microscope_Objective_Magnification':20,
    #'Microscope_Objective_NA':,
    'Microscope_Pixel_Size': .3353,
    'Image_Bit_Depth': 16,
    'Image_Size_X':2160,
    'Image_Size_Y':2160,
    'Timepoint_Primary_Treatment':24,
    #'Timepoint_Secondary_Treatment':0,
    #'Timepoint_Acquisition':0,
    'Treatment_Category':'miRNA',
}

In [5]:
# excitation/emission values used for acquisition
# if value is unknown, define value with np.nan
# if excitation is with a laser (negligible width), 'ex_width' = "None"

# 'fluorophore' is fluorophore conjugated to label or dye variant
# "None" if none, np.nan if unknown
ex_em_fluor_dict = {
    'DNA':{'ex_peak':387,
           'ex_width':np.nan,
           'em_peak':447,
           'em_width':np.nan,
           'fluorophore':'33342'},
    'ER':{'ex_peak':462,
          'ex_width':np.nan,
           'em_peak':520,
           'em_width':np.nan,
           'fluorophore':'Alexa Fluor 488'},
    'RNA':{'ex_peak':531,
           'ex_width':np.nan,
           'em_peak':593,
           'em_width':np.nan,
           'fluorophore':'14'},
    'AGP':{'ex_peak':562,
           'ex_width':np.nan,
           'em_peak':624,
           'em_width':np.nan,
           'fluorophore':'Alexa Fluor 594, Alexa Fluor 562/624'},
    'Mito':{'ex_peak':628,
            'ex_width':np.nan,
           'em_peak':692,
           'em_width':np.nan,
           'fluorophore':'Deep Red'}
}

# Join CPG Metadata

In [62]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "metadata_harmonized")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "metadata_harmonized")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)


In [ ]:

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/", in_staging=in_staging, profile=profile))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/", in_staging=in_staging, profile=profile))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [7]:
load_data_csvs = []
list_batch_names_from_load_data = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        load_data_csvs.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        list_batch_names_from_load_data.append(path.split("/")[-3])

In [16]:
platemaps = []
barcode_platemap_csvs = []
list_batch_names_from_barcode_platemaps = []
list_platemap_names = []
list_batch_names_from_platemaps = []
external_tsv = []

for path in metadata_paths:
    if path.endswith("barcode_platemap.csv"):
        barcode_platemap_csvs.append(s3_loader.read_s3_file(path, sep = ",",in_staging=in_staging, profile=profile))
        list_batch_names_from_barcode_platemaps.append(path.split("/")[-2])
    if '/platemap/' in path and path.endswith(".txt"):
        platemaps.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))
        list_platemap_names.append(path.split("/")[-1].split(".")[0])
        list_batch_names_from_platemaps.append(path.split("/")[-3])
    if 'external' in path:
        if '.csv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = ",", in_staging=in_staging, profile=profile))
        if '.tsv' in path:
            external_tsv.append(s3_loader.read_s3_file(path, sep = "\t", in_staging=in_staging, profile=profile))

In [17]:
all_load_data_platenames = []
for df in load_data_csvs:
    all_load_data_platenames.extend(df['Metadata_Plate'].unique())
all_barcode_platemap_platenames = []
for df in barcode_platemap_csvs:
    all_barcode_platemap_platenames.extend(df['Plate_Map_Name'].unique())
if set(all_load_data_platenames) != set(all_barcode_platemap_platenames):
    print("Plate names in load_data.csv and barcode_platemap.csv do not match.")
    print('Do NOT proceed until they are matched')

In [47]:
complete_df = project.run_conversion(
    load_data_csvs = load_data_csvs, 
    load_data_csv_batch = list_batch_names_from_load_data, 
    platemap_csvs = barcode_platemap_csvs,  
    platemap_csv_batch = list_batch_names_from_barcode_platemaps, 
    platemap_txt= platemaps, 
    platemap_txt_batch = list_platemap_names,
    platemap_txt_name = list_batch_names_from_platemaps,
    external_tsv = external_tsv,
    external_merge_regex = [["compound"],["compound"]]
)

No external metadata found. Not all projects have external metadata.
Merging in barcode platemap csvs
Merging in platemaps
Skipping external metadata. Not all projects have external metadata.
No concentration columns found — skipping concentration merge.
Harmonizing values of Label


In [49]:
complete_df['Cell_Line_Name'] = complete_df['Cell_Line_Name'].str.replace('hek', 'HEK293T')
complete_df['Cell_Line_Name'] = complete_df['Cell_Line_Name'].str.replace('shsy5y', 'SH-SY5Y')

In [50]:
# TODO need to figure out where duplication is coming from in code and actually fix it
complete_df['Batch'] = complete_df['Batch_x']

# Additional cleaning steps

In [51]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

all_cell_lines = inferable_metadata['Cell_Line_Name'].keys()
correct_line_name = {}
for line in all_cell_lines:
    correct_line_name[line.lower()] = line
complete_df['Cell_Line_Name'] = (
    complete_df['Cell_Line_Name']
    .str.lower()
    .map(correct_line_name)
)

for column in inferable_metadata:
    if column == 'Label':
        if per_dataset_manual['CP_Version'] == 'other':
            print("Did not infer label metadata because CP_Version not inferrable")
            print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")
            continue
        else:
            if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
                print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
                print(f"Current labels are {complete_df['Label'].unique()}")
                continue
    # infer metadata from known relationships
    for entry in inferable_metadata[column]:
        for inferred_column in inferable_metadata[column][entry]:
            if not inferred_column in complete_df.columns:
                complete_df[inferred_column] = np.nan
                complete_df[inferred_column] = complete_df[inferred_column].astype('str')
            complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]


In [52]:
# add per label excitation and emission values
# can delete if all values are unknown
if set(complete_df['Label'].unique()) == ex_em_fluor_dict.keys():
    for key in ex_em_fluor_dict:
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Peak"] = ex_em_fluor_dict[key]['ex_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Excitation_Width"] = ex_em_fluor_dict[key]['ex_width']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Peak"] = ex_em_fluor_dict[key]['em_peak']
        complete_df.loc[complete_df["Label"] == key, "Microscope_Emission_Width"] = ex_em_fluor_dict[key]['em_width']
        complete_df.loc[complete_df["Label"] == key, "Label_Fluorophore"] = ex_em_fluor_dict[key]['fluorophore']
else:
    print("Excitation/Emission dictionary does not match Labels")
    print(f"Available labels are {set(complete_df['Label'].unique())}")
    print(f"Defined keys are {ex_em_fluor_dict.keys()}")

In [53]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [54]:
# cleanup
complete_df['Site'] = [x.replace('s','') for x in complete_df['Site']]

In [55]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

Column Label_Fluorophore is not harmonized, <StringArray>
['Alexa Fluor 594, Alexa Fluor 562/624']
Length: 1, dtype: string not in values
Removing columns not in ontology: ['PlateLayout', 'Batch_y', 'Label', 'Plate_Map', 'Batch_x', 'Replicate', 'User_Name']
Adding missing ontology columns: ['Microscope_Objective_NA', 'Treatment_Concentration', 'Treatment_PubChem_CID', 'Cell_Line_Modification', 'Timepoint_Acquisition', 'Image_Position_Z', 'Treatment_Broad_Sample', 'Treatment_InChIKey', 'Treatment_Control_Class', 'Timepoint_Secondary_Treatment', 'Treatment_Mechanism', 'Treatment_Solvent', 'Treatment_Secondary_Treatment', 'Treatment_SMILES']
View of data that will be kept


,Plate,Well,Site,File Path,File Name,Cell_Line_Name,Treatment_Primary_Treatment,Batch,Label_Reagent,Label_Structure,...,Microscope_Binning,Microscope_Modality,Microscope_Objective_Magnification,Microscope_Pixel_Size,Image_Bit_Depth,Image_Size_X,Image_Size_Y,Timepoint_Primary_Treatment,Treatment_Category,Source
0,P10_12269,A01,1,https://cellpainting-gallery.s3.us-east-1.amaz...,HEK_A01_s1_w156C93A25-7F38-4EED-9934-585D0359A9CF,HEK293T,hsa-miR-5704,20190704_hek_Replicate1,Hoechst,Nucleus,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
1,P10_12269,A01,2,https://cellpainting-gallery.s3.us-east-1.amaz...,HEK_A01_s2_w198E148DA-4CDA-48E6-9D98-D33ECD4936BB,HEK293T,hsa-miR-5704,20190704_hek_Replicate1,Hoechst,Nucleus,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
2,P10_12269,A01,3,https://cellpainting-gallery.s3.us-east-1.amaz...,HEK_A01_s3_w19572F3D9-8EBB-4FFC-9B27-8602F36DDC8E,HEK293T,hsa-miR-5704,20190704_hek_Replicate1,Hoechst,Nucleus,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
3,P10_12269,A01,4,https://cellpainting-gallery.s3.us-east-1.amaz...,HEK_A01_s4_w1A6E6126B-B56D-4AF5-8107-2BA525D5D76F,HEK293T,hsa-miR-5704,20190704_hek_Replicate1,Hoechst,Nucleus,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
4,P10_12269,A02,1,https://cellpainting-gallery.s3.us-east-1.amaz...,HEK_A02_s1_w1E7605B40-B780-4285-A9EF-9D5BCB935A57,HEK293T,hsa-miR-5704,20190704_hek_Replicate1,Hoechst,Nucleus,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1036795,P9_12559,P23,4,https://cellpainting-gallery.s3.us-east-1.amaz...,N2As_P23_s4_w58E160E2E-5B3D-4374-B79F-DEF4A14E...,N2a,control_tr_dmso,20190922_n2a_Replicate2,Mitotracker,Mitochondria,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
1036796,P9_12559,P24,1,https://cellpainting-gallery.s3.us-east-1.amaz...,N2As_P24_s1_w5CE436047-DC11-4C7E-A42A-06F68E76...,N2a,control_tr_dmso,20190922_n2a_Replicate2,Mitotracker,Mitochondria,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
1036797,P9_12559,P24,2,https://cellpainting-gallery.s3.us-east-1.amaz...,N2As_P24_s2_w55E6C2285-84F0-4DAF-9E40-67EAB081...,N2a,control_tr_dmso,20190922_n2a_Replicate2,Mitotracker,Mitochondria,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc
1036798,P9_12559,P24,3,https://cellpainting-gallery.s3.us-east-1.amaz...,N2As_P24_s3_w5A0D472BE-B6B0-4DB4-B460-E4246245...,N2a,control_tr_dmso,20190922_n2a_Replicate2,Mitotracker,Mitochondria,...,1,Widefield,20,0.3353,16,2160,2160,24.0,miRNA,edinburgh_igc


In [59]:
print("View of data that will be removed")
complete_df[extra_cols]

View of data that will be removed


,PlateLayout,Batch_y,Label,Plate_Map,Batch_x,Replicate,User_Name
0,10,P10_12269,DNA,P10_12269,20190704_hek_Replicate1,1,URL_OrigDNA
1,10,P10_12269,DNA,P10_12269,20190704_hek_Replicate1,1,URL_OrigDNA
2,10,P10_12269,DNA,P10_12269,20190704_hek_Replicate1,1,URL_OrigDNA
3,10,P10_12269,DNA,P10_12269,20190704_hek_Replicate1,1,URL_OrigDNA
4,10,P10_12269,DNA,P10_12269,20190704_hek_Replicate1,1,URL_OrigDNA
...,...,...,...,...,...,...,...
1036795,9,P9_12559,Mito,P9_12559,20190922_n2a_Replicate2,2,URL_OrigMito
1036796,9,P9_12559,Mito,P9_12559,20190922_n2a_Replicate2,2,URL_OrigMito
1036797,9,P9_12559,Mito,P9_12559,20190922_n2a_Replicate2,2,URL_OrigMito
1036798,9,P9_12559,Mito,P9_12559,20190922_n2a_Replicate2,2,URL_OrigMito


In [60]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

Column Label_Fluorophore is not harmonized, <StringArray>
['Alexa Fluor 594, Alexa Fluor 562/624']
Length: 1, dtype: string not in values
Removing columns not in ontology: ['PlateLayout', 'Batch_y', 'Label', 'Plate_Map', 'Batch_x', 'Replicate', 'User_Name']
Adding missing ontology columns: ['Microscope_Objective_NA', 'Treatment_Concentration', 'Treatment_PubChem_CID', 'Cell_Line_Modification', 'Timepoint_Acquisition', 'Image_Position_Z', 'Treatment_Broad_Sample', 'Treatment_InChIKey', 'Treatment_Control_Class', 'Timepoint_Secondary_Treatment', 'Treatment_Mechanism', 'Treatment_Solvent', 'Treatment_Secondary_Treatment', 'Treatment_SMILES']


In [63]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata_v0_1.parquet")
complete_df.to_parquet(saved_path)